# 모델 학습: YOLO26 디텍터 + ResNet18 분류기 2종

앞서 만든 세 데이터셋을 각각 학습합니다.

| 모델 | 데이터 | 클래스 | 용도 |
|---|---|---|---|
| YOLO26 (Ultralytics) | `rummikub_data_3` | 53 (색+숫자 조합) | 전체 사진에서 타일 위치+클래스 탐지 |
| ResNet18 | `classifier_number` | 14 (1~13, joker) | crop된 타일의 숫자 분류 |
| ResNet18 | `classifier_color` | 5 (black/blue/orange/red/joker) | crop된 타일의 색상 분류 |

**경로 가정**: 이전 노트북들과 동일하게 현재 작업 디렉터리가
`.../boardgame-ai/rummikub_0730`라고 가정하고 전부 상대경로로 씁니다. 다르면 아래 설정 셀의
경로 3개(`RUMMIKUB_DATA_3`, `CLASSIFIER_NUMBER`, `CLASSIFIER_COLOR`)만 바꾸면 됩니다.

**YOLO 모델 크기**: "yolo26v"는 Ultralytics YOLO26의 크기 변형(n/s/m/l/x)을 뜻하는 것으로 보고
`YOLO_MODEL_SIZE` 변수로 뒀습니다(기본값 `n`, 가장 가볍고 빠름). 로컬에 이미 학습된 `best.pt`가
있다면 `YOLO_MODEL_NAME`에 그 경로를 직접 넣어도 됩니다.

**출력**: 세 가중치 파일 모두 `weights/` 폴더에 저장됩니다
(`yolo26_detector_best.pt`, `resnet18_number_best.pt`, `resnet18_color_best.pt`).

## 0. (필요시) 패키지 설치
이미 설치되어 있다면 건너뛰세요.

In [ ]:
# !pip install -q ultralytics torch torchvision

## 공통 설정

In [ ]:
from __future__ import annotations

import random
import shutil
from pathlib import Path

import torch

# ---- 경로 ----
RUMMIKUB_DATA_3 = Path("rummikub_data_3")
CLASSIFIER_NUMBER = Path("classifier_number")
CLASSIFIER_COLOR = Path("classifier_color")
WEIGHTS_OUT = Path("weights")
WEIGHTS_OUT.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# ---- rummikub/classes.py의 53클래스 정의를 그대로 옮겨온 것 (외부 의존성 없음) ----
COLORS = ("black", "blue", "orange", "red")


def _raw_label(canonical_name: str) -> str:
    if canonical_name == "joker":
        return "Joker"
    color, number = canonical_name.rsplit("_", 1)
    return f"{color.capitalize()}{number}"


_NUMERIC_ORDER = [f"{color}_{number}" for color in COLORS for number in range(1, 14)] + ["joker"]
CLASS_NAMES = sorted(_NUMERIC_ORDER, key=_raw_label)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## 1. YOLO26 디텍터 학습 (`rummikub_data_3`)

`rummikub_data_3`는 `train`/`test`만 있고 `val`이 없으므로, `train`의 10%를 무작위로 떼어
학습 중 조기 종료/모델 선택용 val로 씁니다. `test`는 최종 평가용으로 전혀 건드리지 않습니다.
파일을 복사하지 않고 이미지 경로 목록(txt)만 만들어 Ultralytics에 넘깁니다.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

train_image_dir = RUMMIKUB_DATA_3 / "images" / "train"
all_train_images = sorted(
    p for p in train_image_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES
)
assert all_train_images, f"이미지가 없습니다: {train_image_dir.resolve()}"

shuffled = all_train_images.copy()
random.Random(RANDOM_SEED).shuffle(shuffled)
VAL_RATIO = 0.1
n_val = max(1, int(len(shuffled) * VAL_RATIO))
val_subset = shuffled[:n_val]
train_subset = shuffled[n_val:]
print(f"train {len(train_subset)}장 / val(train에서 분리) {len(val_subset)}장")

yolo_dir = Path("yolo_dataset")
yolo_dir.mkdir(exist_ok=True)
(yolo_dir / "train.txt").write_text("\n".join(str(p.resolve()) for p in train_subset))
(yolo_dir / "val.txt").write_text("\n".join(str(p.resolve()) for p in val_subset))

names_yaml = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES))
dataset_yaml_text = (
    f"train: {(yolo_dir / 'train.txt').resolve()}\n"
    f"val: {(yolo_dir / 'val.txt').resolve()}\n"
    f"test: {(RUMMIKUB_DATA_3 / 'images' / 'test').resolve()}\n"
    "\n"
    "names:\n"
    f"{names_yaml}\n"
)
dataset_yaml_path = yolo_dir / "dataset.yaml"
dataset_yaml_path.write_text(dataset_yaml_text, encoding="utf-8")
print(dataset_yaml_text)

In [ ]:
from ultralytics import YOLO

YOLO_MODEL_SIZE = "n"  # n / s / m / l / x
YOLO_MODEL_NAME = f"yolo26{YOLO_MODEL_SIZE}.pt"  # 로컬 best.pt가 있으면 이 줄을 그 경로로 교체
YOLO_EPOCHS = 100
YOLO_IMGSZ = 960

yolo_model = YOLO(YOLO_MODEL_NAME)
yolo_model.train(
    data=str(dataset_yaml_path.resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=-1,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    patience=25,
    project="runs/detect",
    name="rummikub_yolo26",
    pretrained=True,
    seed=RANDOM_SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
)

yolo_best_path = Path(yolo_model.trainer.best)
yolo_out_path = WEIGHTS_OUT / "yolo26_detector_best.pt"
shutil.copy2(yolo_best_path, yolo_out_path)
print("YOLO 가중치 저장:", yolo_out_path.resolve())

## 2. ResNet18 분류기 공통 학습 함수
숫자/색상 분류기 둘 다 같은 구조(ImageNet 사전학습 ResNet18 + 마지막 fc만 클래스 수에 맞게 교체)라
함수 하나로 재사용합니다. `classifier_number`, `classifier_color` 둘 다 이미
`{train,val,test}/{class}/` 구조로 나뉘어 있으므로 `torchvision.datasets.ImageFolder`를 그대로 씁니다.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

IMG_SIZE = 96
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# 이미 회전(0/90/180/270) x 밝기(-25/+25) crop 증강이 되어 있는 데이터이므로,
# 여기서 추가로 flip/rotate 증강을 하면 과할 수 있어 resize+정규화만 적용합니다.
train_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def build_resnet_classifier(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


def train_classifier(data_root: Path, output_filename: str):
    train_ds = datasets.ImageFolder(data_root / "train", transform=train_tf)
    val_ds = datasets.ImageFolder(data_root / "val", transform=eval_tf)
    test_ds = datasets.ImageFolder(data_root / "test", transform=eval_tf)
    assert train_ds.classes == val_ds.classes == test_ds.classes, (
        "train/val/test의 클래스 폴더 목록이 다릅니다 — 데이터셋 생성 노트북을 다시 확인하세요."
    )
    print(f"[{output_filename}] classes({len(train_ds.classes)}): {train_ds.classes}")
    print(
        f"[{output_filename}] train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}"
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    model = build_resnet_classifier(len(train_ds.classes))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        train_loss = running_loss / len(train_ds)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total if total else 0.0
        print(
            f"[{output_filename}] epoch {epoch}/{NUM_EPOCHS} "
            f"train_loss={train_loss:.4f} val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    test_acc = correct / total if total else 0.0

    out_path = WEIGHTS_OUT / output_filename
    torch.save({"state_dict": model.state_dict(), "classes": train_ds.classes}, out_path)
    print(
        f"[{output_filename}] best_val_acc={best_val_acc:.4f} test_acc={test_acc:.4f} "
        f"-> 저장: {out_path.resolve()}"
    )
    return {"best_val_acc": best_val_acc, "test_acc": test_acc, "path": out_path}

## 3. 숫자 분류기 학습 (`classifier_number`, 14클래스)

In [ ]:
number_result = train_classifier(CLASSIFIER_NUMBER, "resnet18_number_best.pt")

## 4. 색상 분류기 학습 (`classifier_color`, 5클래스)

In [ ]:
color_result = train_classifier(CLASSIFIER_COLOR, "resnet18_color_best.pt")

## 5. 요약

In [ ]:
print("YOLO26 detector:", yolo_out_path.resolve())
print(
    "ResNet18 number:", number_result["path"].resolve(),
    f"(val_acc={number_result['best_val_acc']:.4f}, test_acc={number_result['test_acc']:.4f})",
)
print(
    "ResNet18 color:", color_result["path"].resolve(),
    f"(val_acc={color_result['best_val_acc']:.4f}, test_acc={color_result['test_acc']:.4f})",
)